# III-2 Google Colab 入門（ノートブック版）

教科書の III-2 の練習用の学習 `train_toy.py`（合成のモータ音の正常／異常を1次元CNNで分類。チェックポイントから正確に再開できる）と、結果を比べる `compare_runs.py` を、上から順に実行すれば動く1冊にしたものです。**Google Colab（GPU でも CPU でも）**でも、Mac の Jupyter（MPS か CPU）でも動きます。執筆時の Mac（Apple シリコン、MPS）で全部で約2分、CPU だけなら4〜5分です。

| 節 | 試すこと | 教科書の手順 | 使うファイル |
|---|---|---|---|
| 0 | GPU の確認（セル1）、保存先を決める（セル2。Drive は任意） | 手順 7〜8 | ― |
| 1 | 小さく動作確認する（`--quick`） | 手順 2 | `train_toy.py` |
| 2 | 中断と再開（Ctrl-C の代わりに SIGINT を送る）と、止めなかったときとの一致 | 手順 3・10 | `train_toy.py` |
| 3 | 比較用の実行（本番に近い条件を小さくしたもの） | 手順 4・9 | `train_toy.py` |
| 4 | 実行どうしの処理速度を比べる（疑似反復の注意つき） | 手順 12 | `compare_runs.py` |
| 5 | チェックポイントの間隔：Young の近似式とシミュレーション | III-2-3-2、発展課題 | ― |
| 6 | 結果の確認とランタイムの解放（セル6・7） | 手順 11 | ― |

- **教科書のやり方との違い**：教科書では、ノートブックは「呼び出すだけ」にして、コードは GitHub の private リポジトリから clone します（III-2-3-3・セル3）。このノートブックは教科書の HTML と一緒に配るものなので、clone の代わりに `train_toy.py`・`compare_runs.py` を `%%writefile` で書き出します（**中身は教科書と同じ**です。変えたのは `compare_runs.py` の図の日本語フォントの候補を足したことだけ）。トークンもシークレットも使いません。自分のリポジトリで回すときは、下に載せた教科書のセル1〜7 をそのまま使ってください。
- ターミナルの `uv run python train_toy.py …`（Colab の `!python train_toy.py …`）の代わりに、`run("train_toy.py …")` で同じ引数を渡して別のプロセスで実行します。
- **既定は小さい設定**です。3 節は教科書の `--epochs 4 --n-train 8000` の代わりに `--epochs 3 --n-train 4000` にしています（本番の設定はセルのコメントにあります）。`--workers` は、Colab では教科書の既定の 2、それ以外（Mac の Jupyter）では 0 にしています（`WORKERS` の行）。
- `ok()` は、教科書の「確かめること」のうち、機械に依らず成り立つこと（ファイルができる、`--quick` の2エポック目の val_acc が 0.9 以上、再開したエポックの `resumed` が True、止めなかった実行と結果がほぼ同じ、Young の式の値）だけを確かめます。速さ（サンプル/秒）は機械で大きく変わるので表示だけです。
- 保存先は、既定ではこのノートブックの隣の `runs/` です。Colab で `/content` に保存すると、ランタイムを削除したときに消えます。Drive に残したいときは、0 節の `USE_DRIVE = True` にしてください（Drive のマウントを許可する画面が出ます）。
- **入れていないもの**：GitHub のトークンを作ってシークレットに登録すること、private リポジトリの clone（手順 5〜8 のうちリポジトリの部分）、Drive for desktop で結果を Mac に持ってくること、本物の切断（「ランタイムを接続解除して削除」。2 節で同じことを SIGINT で試します）。
- ⚠ 研究室の SEM 画像や録音などを、許可なく個人の Google Drive に置かないでください。このノートブックが使うのは、その場で作る合成の信号だけです。
- Colab では、最初のほうの「0-1. 図の日本語のフォント」のセルが日本語のフォント（Noto Sans CJK JP）を入れます（数十秒〜1分ほど。Mac では Hiragino Sans を使うので何もしません）。そのセルが「★日本語のフォントが見つからない」と表示したときは、図の日本語が四角（□）になります（数値の確認には影響しません）。

In [ ]:
# 0. 準備：足りないパッケージだけ入れる（Colab には torch・numpy・pandas・scipy・matplotlib が最初から入っている。入れ直さない）
import importlib, subprocess, sys
for mod, pkg in [("torch", "torch"), ("numpy", "numpy"), ("pandas", "pandas"), ("scipy", "scipy"), ("matplotlib", "matplotlib")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import os, shutil, torch, platform
IN_COLAB = "google.colab" in sys.modules
print("OK Python", platform.python_version(), "/ torch", torch.__version__, "/ CUDA:", torch.cuda.is_available(),
      "/ MPS:", torch.backends.mps.is_available(), "/ Colab:", IN_COLAB)
if shutil.which("nvidia-smi"):                     # 教科書のセル1（!nvidia-smi）と同じ
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
else:
    print("nvidia-smi がない（GPU のランタイムではない）。CPU か MPS で動きます")
print("（教科書は torch 2.14.1 で確かめた。Colab の torch の版は違うことがある）")

In [ ]:
# 0-1. 準備：図の日本語のフォント（Colab のときだけ fonts-noto-cjk を入れる。Mac は Hiragino Sans を使うので何もしない）
import glob, os, subprocess, sys
import matplotlib
from matplotlib import font_manager

JP_FONTS = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic"]   # 本文の図と同じ候補（上から順に使う）


def find_jp_font():
    have = {f.name for f in font_manager.fontManager.ttflist}
    return next((n for n in JP_FONTS if n in have), None)


IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and find_jp_font() is None:
    print("Colab：日本語のフォント（fonts-noto-cjk）を入れます（数十秒〜1分ほど）")
    cmd = ["apt-get", "install", "-y", "-qq", "fonts-noto-cjk"]
    if subprocess.run(cmd, capture_output=True).returncode != 0:
        subprocess.run(["apt-get", "update", "-qq"], capture_output=True)   # パッケージの一覧が古いときは、更新してもう一度
        subprocess.run(cmd, capture_output=True)
    for path in glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc"):
        font_manager.fontManager.addfont(path)   # このノートブックの中でかく図に効く（再起動は要らない）
    # 別のプロセスで動かすスクリプト（subprocess や !python）にも見えるように、フォントの一覧のキャッシュを書き直す
    font_manager.json_dump(font_manager.fontManager, os.path.join(
        matplotlib.get_cachedir(), f"fontlist-v{font_manager.FontManager.__version__}.json"))

JP_FONT = find_jp_font()
if JP_FONT:
    matplotlib.rcParams["font.family"] = [JP_FONT, "sans-serif"]
    matplotlib.rcParams["font.sans-serif"] = [JP_FONT] + [f for f in matplotlib.rcParams["font.sans-serif"] if f != JP_FONT]
    if IN_COLAB:   # 本文のスクリプトの "sans-serif" も日本語のフォントになるように、Colab の matplotlib の設定ファイルに書く
        rc = os.path.join(matplotlib.get_configdir(), "matplotlibrc")
        lines = open(rc, encoding="utf-8").read().splitlines() if os.path.exists(rc) else []
        lines = [l for l in lines if not l.startswith("font.sans-serif")]
        lines.append("font.sans-serif: " + ", ".join(matplotlib.rcParams["font.sans-serif"]))
        with open(rc, "w", encoding="utf-8") as f:
            f.write("\n".join(lines) + "\n")
    print("図の日本語のフォント：", JP_FONT)
else:
    print("★日本語のフォントが見つからないので、図の日本語が四角（□）になります（数値には影響しません）。")
    print("  Colab なら、インターネットにつながっているかを確かめて、このセルをもう一度実行してください。"
          "Linux の Jupyter なら、sudo apt install fonts-noto-cjk で入れてから、カーネルを再起動します。")

In [ ]:
# 教科書のセル2 に当たるところ：保存先を決める（既定はこのノートブックの隣の runs/）
USE_DRIVE = False                                 # Colab で Drive に保存したいときだけ True にする
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RUNS = "/content/drive/MyDrive/home-lab/runs"
else:
    RUNS = "runs"
os.makedirs(RUNS, exist_ok=True)
WORKERS = 2 if IN_COLAB else 0                    # 教科書の既定は 2。Mac の Jupyter では 0（spawn のトラブルを避け、ほかの作業の邪魔をしない）
print("保存先:", os.path.abspath(RUNS), " --workers", WORKERS)

**教科書のリポジトリ構成（~/home-cloud。読むだけ）**

```
home-cloud/                 ← GitHub の private リポジトリ
├── .gitignore               ← runs/ data/ *.pt .env などを除外（III-1）
├── .env.example
├── pyproject.toml / uv.lock ← uv が作る（III-4 で詳しく）
├── requirements-colab.txt   ← Colab で追加するものだけ
├── bench_step.py            ← III-1
├── cost_estimator.py        ← III-1
├── buy_vs_rent.py           ← III-1
├── ledger_summary.py        ← III-1
├── cost_ledger.csv          ← III-1（費用の台帳。金額だけで秘密は書かない）
├── train_toy.py             ← III-2：どこでも動く練習用の学習
├── compare_runs.py          ← III-2：Mac で結果を比べる
└── runs/                    ← 結果（Git には入れない。Drive・rsync で運ぶ）
```

**requirements-colab.txt（Colab に追加で入れるものだけ。読むだけ）**

```
# Colab にあらかじめ入っている torch / numpy / pandas などは入れ直さない（入れ直すと壊れることがある）。
# ここには「Colab に無くて、自分のコードが使うもの」だけを書く。
# 版の固定（==）は III-4 で扱う。動いた版を run_info.json と合わせて記録しておく。
# 下は書き方の例（train_toy.py は使わない。第II部のウェーブレットを Colab で使うときに必要になる）
PyWavelets
```

**教科書の Colab のセル1〜7（自分の private リポジトリで回すとき用。このノートブックでは実行しません）**

```python
# ===== セル1：GPU の確認 =====
!nvidia-smi
import torch, sys
print("Python", sys.version.split()[0], "/ torch", torch.__version__, "/ CUDA:", torch.cuda.is_available())

# ===== セル2：Google Drive をマウント（チェックポイントと結果の保存先） =====
from google.colab import drive
drive.mount("/content/drive")
RUN_DIR = "/content/drive/MyDrive/home-lab/runs/colab_toy01"
import os
os.makedirs(RUN_DIR, exist_ok=True)
print("保存先:", RUN_DIR)

# ===== セル3：private リポジトリを clone（トークンはシークレットから読む。表示しない） =====
import subprocess
from google.colab import userdata
REPO = "YOURNAME/home-cloud"             # ← 自分のリポジトリ名
DEST = "/content/home-cloud"
if not os.path.exists(DEST):
    token = userdata.get("GH_TOKEN")
    r = subprocess.run(["git", "clone", "--depth", "1",
                        f"https://{token}@github.com/{REPO}.git", DEST],
                       capture_output=True, text=True)
    del token
    if r.returncode != 0:
        # エラー文にトークン入りの URL が含まれうるので、そのまま表示しない
        print("clone に失敗しました（returncode =", r.returncode, "）。リポジトリ名・トークンの権限と期限を確認してください")
    else:
        # .git/config に残ったトークンを消す
        subprocess.run(["git", "-C", DEST, "remote", "set-url", "origin",
                        f"https://github.com/{REPO}.git"], check=True)
        print("clone しました")
!git -C /content/home-cloud log -1 --oneline
!git -C /content/home-cloud remote -v

# ===== セル4：作業フォルダへ移動し、足りないものだけ入れる =====
%cd /content/home-cloud
!pip install -q -r requirements-colab.txt
# （データがある場合の例）Drive から1つのアーカイブとしてコピーし、ローカルで展開する
# !cp /content/drive/MyDrive/home-lab/data/dataset.tar.gz /content/ && tar -xzf /content/dataset.tar.gz -C /content/

# ===== セル5：学習（途中で切れても、同じセルを実行すれば再開する） =====
!python train_toy.py --epochs 4 --n-train 8000 --ckpt-dir "$RUN_DIR" --ckpt-every-min 5 --max-hours 2

# ===== セル6：結果の確認 =====
import pandas as pd
m = pd.read_csv(f"{RUN_DIR}/metrics.csv")
print(m[["epoch", "train_loss", "val_acc", "samples_per_s", "resumed"]])

# ===== セル7：ランタイムを解放する（GPU の割り当てを返す。有料ならユニットの消費も止まる） =====
# 結果が Drive にあることを確認してから実行する
from google.colab import runtime
runtime.unassign()
```

次の2つのセルで、III-2-6 の `train_toy.py` と `compare_runs.py` を書き出します。

In [ ]:
%%writefile train_toy.py
#!/usr/bin/env python3
"""III-2：どこでも動く練習用の学習スクリプト
合成した「モータ音」の正常／異常を、1次元CNNで分類する。
- device を自動で選ぶ（cuda → mps → cpu）。同じコードが Mac と Colab で動く
- 一定時間ごと＋エポックの終わりにチェックポイントを保存。切断されても途中から正確に再開できる
- CUDA では混合精度（AMP）を使う
- --max-hours を過ぎたら、保存して自分で終了する（課金の止め忘れ対策）
使い方:
  uv run python train_toy.py --quick --ckpt-dir runs/mac_quick              # Mac で動作確認
  python train_toy.py --ckpt-dir /content/drive/MyDrive/home-lab/runs/toy01    # Colab で本番
"""
import argparse
import contextlib
import csv
import json
import math
import os
import platform
import random
import signal
import subprocess
import sys
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset

FS = 16000
stop_requested = False


def on_signal(signum, frame):
    global stop_requested
    stop_requested = True
    print(f"\nシグナル {signum} を受信：区切りのよいところで保存して終了します", flush=True)


# ---------------------------------------------------------------- 環境
def pick_device(name="auto"):
    if name != "auto":
        return torch.device(name)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def device_name(dev):
    if dev.type == "cuda":
        return torch.cuda.get_device_name(0)
    if dev.type == "mps":
        return "Apple MPS"
    return platform.processor() or platform.machine()


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def git_commit():
    try:
        r = subprocess.run(["git", "rev-parse", "--short", "HEAD"],
                           capture_output=True, text=True, timeout=5)
        return r.stdout.strip() or "unknown"
    except Exception:
        return "unknown"


def amp_ctx(enabled):
    return torch.autocast("cuda", dtype=torch.float16) if enabled else contextlib.nullcontext()


# ---------------------------------------------------------------- データ（合成）
def make_signal(rng, length, anomalous):
    """基本波 80〜120 Hz と高調波＋雑音。異常は「周期的なクリック」か「高い周波数のうなり」"""
    t = np.arange(length) / FS
    f0 = rng.uniform(80, 120)
    x = np.zeros(length)
    for k in range(1, 6):
        x += np.sin(2 * np.pi * k * f0 * t + rng.uniform(0, 2 * np.pi)) / k
    x += rng.normal(0, 0.3, length)
    if anomalous:
        if rng.integers(2) == 0:
            period = int(FS / rng.uniform(15, 40))
            win = np.hanning(16) * rng.uniform(2, 4)
            for p in range(int(rng.integers(period)), length, period):
                seg = x[p:p + 16]
                seg += win[:len(seg)]
        else:
            fm = rng.uniform(1500, 3000)
            env = 1 + 0.5 * np.sin(2 * np.pi * rng.uniform(5, 20) * t)
            x += 0.6 * np.sin(2 * np.pi * fm * t) * env
    return (x / x.std()).astype(np.float32)


class SynthMotor(Dataset):
    """番号 i から決まる乱数で毎回同じ信号を作る（ファイルに保存しないのでディスクを使わない）"""
    def __init__(self, n, length, seed):
        self.n, self.length, self.seed = n, length, seed

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        rng = np.random.default_rng([self.seed, i])
        y = i % 2                                 # 正常と異常を半分ずつ
        x = make_signal(rng, self.length, bool(y))
        return torch.from_numpy(x).unsqueeze(0), y


# ---------------------------------------------------------------- モデル
class AudioCNN(nn.Module):
    def __init__(self, w=32, ncls=2):
        super().__init__()
        layers, cin = [], 1
        for cout in (w, w, 2 * w, 2 * w, 4 * w, 4 * w):
            layers += [nn.Conv1d(cin, cout, 9, stride=2, padding=4),
                       nn.BatchNorm1d(cout), nn.ReLU(inplace=True)]
            cin = cout
        self.features = nn.Sequential(*layers)
        self.head = nn.Linear(cin, ncls)

    def forward(self, x):
        return self.head(self.features(x).mean(dim=-1))


# ---------------------------------------------------------------- チェックポイント
def save_ckpt(path, state):
    tmp = path + ".tmp"
    torch.save(state, tmp)
    os.replace(tmp, path)                         # 書き込み途中で切れても壊れない


@torch.no_grad()
def evaluate(model, loader, dev, use_amp):
    model.eval()
    correct = n = 0
    for x, y in loader:
        x, y = x.to(dev), y.to(dev)
        with amp_ctx(use_amp):
            pred = model(x).argmax(1)
        correct += (pred == y).sum().item()
        n += len(y)
    model.train()
    return correct / n


def parse():
    ap = argparse.ArgumentParser()
    ap.add_argument("--epochs", type=int, default=10)
    ap.add_argument("--n-train", type=int, default=20000)
    ap.add_argument("--n-val", type=int, default=2000)
    ap.add_argument("--length", type=int, default=4096)
    ap.add_argument("--batch", type=int, default=128)
    ap.add_argument("--lr", type=float, default=1e-3)
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--device", default="auto")
    ap.add_argument("--no-amp", action="store_true")
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--ckpt-dir", default="runs/toy")
    ap.add_argument("--ckpt-every-min", type=float, default=5.0)
    ap.add_argument("--max-hours", type=float, default=3.0, help="この時間を過ぎたら保存して終了")
    ap.add_argument("--quick", action="store_true", help="小さい設定で動作確認")
    a = ap.parse_args()
    if a.quick:
        a.n_train, a.n_val, a.epochs = 4000, 400, 2
    return a


def main():
    a = parse()
    signal.signal(signal.SIGTERM, on_signal)
    signal.signal(signal.SIGINT, on_signal)
    set_seed(a.seed)
    dev = pick_device(a.device)
    use_amp = dev.type == "cuda" and not a.no_amp
    if dev.type == "cuda":
        torch.backends.cudnn.benchmark = True
    os.makedirs(a.ckpt_dir, exist_ok=True)
    last = os.path.join(a.ckpt_dir, "last.pt")
    metrics_path = os.path.join(a.ckpt_dir, "metrics.csv")
    t_start = time.time()

    train_ds = SynthMotor(a.n_train, a.length, seed=a.seed)
    val_ds = SynthMotor(a.n_val, a.length, seed=a.seed + 10_000)
    pin = dev.type == "cuda"
    val_loader = DataLoader(val_ds, batch_size=a.batch * 2, num_workers=a.workers, pin_memory=pin)

    model = AudioCNN().to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=a.lr)
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    start_epoch, start_step = 0, 0
    run = {"loss_sum": 0.0, "n": 0, "t": 0.0}
    resumed = False
    if os.path.exists(last):
        ck = torch.load(last, map_location="cpu", weights_only=False)   # 自分で作ったファイルなので可
        if ck.get("done"):
            print(f"この実行はすでに完了しています: {last}")
            return
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["opt"])
        if use_amp and ck.get("scaler"):
            scaler.load_state_dict(ck["scaler"])
        start_epoch, start_step, run = ck["epoch"], ck["step"], ck["run"]
        resumed = True
        print(f"チェックポイントから再開: epoch {start_epoch} step {start_step}", flush=True)

    # 実行の記録（再開のたびに追記）
    info_path = os.path.join(a.ckpt_dir, "run_info.json")
    info = json.load(open(info_path, encoding="utf-8")) if os.path.exists(info_path) else {"sessions": []}
    info["sessions"].append({
        "started": time.strftime("%Y-%m-%dT%H:%M:%S%z"), "git_commit": git_commit(),
        "torch": torch.__version__, "python": sys.version.split()[0],
        "device": dev.type, "device_name": device_name(dev), "amp": use_amp,
        "args": vars(a), "resumed": resumed})
    with open(info_path, "w", encoding="utf-8") as f:
        json.dump(info, f, ensure_ascii=False, indent=2)
    print(f"device = {device_name(dev)}（{dev.type}）, AMP = {use_amp}, commit = {git_commit()}", flush=True)

    def state(epoch, step, run_state, done=False):
        return {"model": model.state_dict(), "opt": opt.state_dict(),
                "scaler": scaler.state_dict(), "epoch": epoch, "step": step,
                "run": run_state, "args": vars(a), "done": done,
                "saved_at": time.strftime("%Y-%m-%dT%H:%M:%S%z")}

    last_save = time.time()
    for epoch in range(start_epoch, a.epochs):
        first = epoch == start_epoch
        if not first:
            run = {"loss_sum": 0.0, "n": 0, "t": 0.0}
        step = start_step if first else 0
        # エポックごとの並び順を「種＋エポック番号」で決める → 途中からでも同じ順番で再開できる
        g = torch.Generator().manual_seed(a.seed * 1000 + epoch)
        order = torch.randperm(len(train_ds), generator=g).tolist()
        loader = DataLoader(Subset(train_ds, order[step * a.batch:]), batch_size=a.batch,
                            shuffle=False, num_workers=a.workers, pin_memory=pin)
        model.train()
        t_seg = time.time()
        for x, y in loader:
            x = x.to(dev, non_blocking=True)
            y = y.to(dev, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with amp_ctx(use_amp):
                loss = F.cross_entropy(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            run["loss_sum"] += loss.item() * len(y)
            run["n"] += len(y)
            step += 1
            over_time = time.time() - t_start > a.max_hours * 3600
            if stop_requested or over_time or time.time() - last_save > a.ckpt_every_min * 60:
                run["t"] += time.time() - t_seg
                t_seg = time.time()
                save_ckpt(last, state(epoch, step, run))
                last_save = time.time()
                print(f"  チェックポイント保存: epoch {epoch} step {step}", flush=True)
                if stop_requested or over_time:
                    why = "シグナル" if stop_requested else f"--max-hours {a.max_hours}"
                    print(f"{why} により終了します。同じコマンドで再開できます。", flush=True)
                    return
        run["t"] += time.time() - t_seg
        val_acc = evaluate(model, val_loader, dev, use_amp)
        row = {"epoch": epoch, "train_loss": run["loss_sum"] / max(run["n"], 1),
               "val_acc": val_acc, "epoch_time_s": run["t"],
               "samples_per_s": run["n"] / max(run["t"], 1e-9),
               "device": dev.type, "device_name": device_name(dev),
               "resumed": resumed and first, "time": time.strftime("%Y-%m-%dT%H:%M:%S%z")}
        new = not os.path.exists(metrics_path)
        with open(metrics_path, "a", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(row.keys()))
            if new:
                w.writeheader()
            w.writerow(row)
        print(f"epoch {epoch}: loss {row['train_loss']:.4f}  val_acc {val_acc:.4f}  "
              f"{row['samples_per_s']:.0f} サンプル/秒", flush=True)
        save_ckpt(last, state(epoch + 1, 0, {"loss_sum": 0.0, "n": 0, "t": 0.0},
                              done=(epoch + 1 == a.epochs)))
        last_save = time.time()
    print("完了しました。", flush=True)


if __name__ == "__main__":     # DataLoader の並列読み込み（macOS の spawn）に必要
    main()

In [ ]:
%%writefile compare_runs.py
#!/usr/bin/env python3
"""III-2：複数の実行結果（metrics.csv）の処理速度と学習曲線を比べる
最初に指定した実行を基準にして、速度の比と 95% 区間を出す。
区間は、1回の実行の中のエポックを反復とみなした「この実行どうし」の目安（エポックは独立な反復ではない）。
マシンや GPU の種類を比べるときは III-3 のとおり、マシンを反復の単位にする。
使い方:
  uv run python compare_runs.py runs/mac_base runs/colab_toy01
  uv run python compare_runs.py runs/mac_base runs/colab_toy01 --price colab_toy01=0
"""
import argparse
import pathlib

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams["font.family"] = ["Hiragino Sans", "Noto Sans CJK JP", "IPAexGothic", "sans-serif"]   # ノートブック版：Colab でも止まらないように候補を足す
plt.rcParams["axes.unicode_minus"] = False


def log_speed(m):
    """最初のエポックはウォームアップ（cudnn の準備など）なので、3エポック以上あれば除く"""
    s = m["samples_per_s"].to_numpy()
    if len(s) >= 3:
        s = s[1:]
    return np.log(s)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("runs", nargs="+")
    ap.add_argument("--price", nargs="*", default=[], help="実行名=円/時（例 colab_toy01=0）")
    a = ap.parse_args()
    price = {k: float(v) for k, v in (p.split("=") for p in a.price)}

    data = {}
    for r in a.runs:
        d = pathlib.Path(r)
        m = pd.read_csv(d / "metrics.csv")
        data[d.name] = m
        ls = log_speed(m)
        gm = np.exp(ls.mean())
        line = f"{d.name:16s} {m['device_name'].iloc[-1]:28s} 処理量 {gm:8.0f} サンプル/秒"
        if len(ls) >= 2:
            t = stats.t.ppf(0.975, len(ls) - 1)
            se = ls.std(ddof=1) / np.sqrt(len(ls))
            line += f"（95%区間 {np.exp(ls.mean()-t*se):.0f}〜{np.exp(ls.mean()+t*se):.0f}、n={len(ls)}）"
        if d.name in price and price[d.name] > 0:
            line += f"  100万サンプルあたり {price[d.name] / (gm * 3600) * 1e6:.1f} 円"
        print(line)

    names = list(data)
    base = log_speed(data[names[0]])
    for name in names[1:]:
        other = log_speed(data[name])
        if len(base) < 2 or len(other) < 2:
            print(f"{name}：エポック数が少ないので区間は出せません（比 {np.exp(other.mean()-base.mean()):.2f} 倍）")
            continue
        # 対数の差に Welch の t 区間 → 指数をとって「速度の比」の区間にする
        diff = other.mean() - base.mean()
        va, vb = base.var(ddof=1) / len(base), other.var(ddof=1) / len(other)
        se = np.sqrt(va + vb)
        dof = (va + vb) ** 2 / (va**2 / (len(base) - 1) + vb**2 / (len(other) - 1))
        t = stats.t.ppf(0.975, dof)
        print(f"{name} は {names[0]} の {np.exp(diff):.2f} 倍速い"
              f"（95%区間 {np.exp(diff - t*se):.2f}〜{np.exp(diff + t*se):.2f} 倍、Welch 自由度 {dof:.1f}）")

    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    for name, m in data.items():
        ax[0].plot(m["epoch"], m["val_acc"], "o-", label=name)
        ax[1].plot(m["epoch"], m["samples_per_s"], "o-", label=name)
    ax[0].set(xlabel="エポック", ylabel="検証データの正解率", title="学習曲線")
    ax[1].set(xlabel="エポック", ylabel="サンプル/秒", title="処理速度", yscale="log")
    for x in ax:
        x.legend()
    fig.tight_layout()
    fig.savefig("compare_runs.png", dpi=150)
    print("compare_runs.png を保存しました")


if __name__ == "__main__":
    main()

In [ ]:
import json, re, shlex, signal, time
import pandas as pd
from IPython.display import Image, display


def run(args):
    """ターミナルの「uv run python ○○.py …」の代わり。表示を出し、あとで確かめられるように文字列で返す"""
    p = subprocess.run([sys.executable] + shlex.split(args), capture_output=True, text=True)
    print(p.stdout, end="")
    err = "\n".join(l for l in p.stderr.splitlines() if "findfont" not in l and "font cache" not in l)
    if err.strip():
        print(err, file=sys.stderr)
    assert p.returncode == 0, "★実行に失敗：" + args
    return p.stdout


def ok(cond, msg):
    assert cond, "★確かめ失敗：" + msg
    print("OK", msg)


def fresh(name):
    """やり直したときのために、前の結果のフォルダを消す（残っていると「すでに完了しています」になる）"""
    d = os.path.join(RUNS, name)
    shutil.rmtree(d, ignore_errors=True)
    return d


print("準備ができました")

## 1. 小さく動作確認する（手順 2、`--quick`）

`--quick` は、学習データ 4000 個・検証データ 400 個・2 エポックです。確かめること（手順 2）：「2エポックで終わり（MPS で1分ほど、CPU で1〜2分）、runs/mac_quick/ に last.pt、metrics.csv、run_info.json ができる。2エポック目の val_acc が0.9以上（1エポック目は0.6前後のことがある。III-2-7）」。

III-2-7 の結果の例：「1エポック目の val_acc は0.6前後、2エポック目で0.98前後になる（筆者の Mac・torch 2.14.1、種0〜2、MPS と CPU で 0.980〜0.990）」。1エポック目が低いのは、BatchNorm の移動平均が追いついていないためです（III-2-7 の表）。

In [ ]:
Q = fresh("nb_quick")                        # 教科書では runs/mac_quick（Colab の GPU なら Drive の runs/… に）
t0 = time.time()
out = run(f"train_toy.py --quick --ckpt-dir {Q} --workers {WORKERS}")
print(f"（{time.time() - t0:.0f} 秒）")
for f in ("last.pt", "metrics.csv", "run_info.json"):
    assert os.path.exists(os.path.join(Q, f)), f
ok(True, f"{Q}/ に last.pt・metrics.csv・run_info.json ができた")
mq = pd.read_csv(os.path.join(Q, "metrics.csv"))
print(mq[["epoch", "train_loss", "val_acc", "samples_per_s", "device"]])
ok(len(mq) == 2 and "完了しました。" in out, "2エポックで終わった")
ok(mq["val_acc"].iloc[1] >= 0.9, f"2エポック目の val_acc = {mq['val_acc'].iloc[1]:.4f}（0.9 以上。本文の目安は 0.98 前後）")
print(f"1エポック目の val_acc = {mq['val_acc'].iloc[0]:.4f}（本文：0.6 前後のことがある。BatchNorm の移動平均のため）")
info = json.load(open(os.path.join(Q, "run_info.json"), encoding="utf-8"))
ok(info["sessions"][0]["args"]["n_train"] == 4000 and info["sessions"][0]["resumed"] is False,
   "run_info.json に、引数（--quick で n_train = 4000）・torch の版・device が記録された")

## 2. 中断と再開（手順 3・10）

教科書の手順 3 では、学習の途中で Ctrl-C を押し、同じコマンドをもう一度実行します（Colab の手順 10 では、ランタイムを接続解除して削除してから、セルを上から実行し直します）。ここでは、別のプロセスで動かした `train_toy.py` に、最初のチェックポイントを保存したところ（または1エポック目が終わったところ）で SIGINT（Ctrl-C と同じシグナル）を送ります。

確かめること：「1回目は『シグナルを受信 → 保存して終了』と表示される。2回目は『チェックポイントから再開: epoch … step …』と表示され、最後まで進む。metrics.csv の該当エポックの resumed が True」。

教科書の手順 3 は `--epochs 4 --n-train 8000 --ckpt-every-min 0.2` ですが、ここでは 1 節と**同じ条件**（`--quick`）で `--ckpt-every-min 0.05`（3 秒ごと）にします。エポックごとの並び順は「種＋エポック番号」で決まるので、途中で止めても、1 節の止めなかった実行とほぼ同じ結果になるはずです（III-2-3-2。GPU・MPS の計算の細かな非決定性による小さな差はありうる）。

In [ ]:
R_ = fresh("nb_resume")                       # 教科書では runs/mac_resume
cmd = f"train_toy.py --quick --ckpt-dir {R_} --workers {WORKERS} --ckpt-every-min 0.05"
p = subprocess.Popen([sys.executable] + shlex.split(cmd), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
log1, sent = [], False
for line in p.stdout:
    print(line, end="")
    log1.append(line)
    if not sent and ("チェックポイント保存" in line or line.startswith("epoch 0:")):
        p.send_signal(signal.SIGINT)              # Ctrl-C と同じ
        sent = True
p.wait()
log1 = "".join(log1)
ok(sent and "シグナル" in log1 and "により終了します" in log1, "1回目：「シグナルを受信 → 保存して終了」と表示された")
print("\n---- 同じコマンドをもう一度 ----")
log2 = run(cmd)
ok(re.search(r"チェックポイントから再開: epoch \d+ step \d+", log2) is not None and "完了しました。" in log2,
   "2回目：「チェックポイントから再開: epoch … step …」と表示され、最後まで進んだ")
mr = pd.read_csv(os.path.join(R_, "metrics.csv"))
print(mr[["epoch", "train_loss", "val_acc", "resumed"]])
ep = int(re.search(r"再開: epoch (\d+)", log2).group(1))
ok(bool(mr.loc[mr["epoch"] == ep, "resumed"].iloc[0]), f"再開したエポック（epoch {ep}）の resumed が True")
d = abs(mr["val_acc"].iloc[-1] - mq["val_acc"].iloc[-1])
ok(d <= 0.02, f"最後の val_acc：止めた実行 {mr['val_acc'].iloc[-1]:.4f}、止めなかった実行（1 節）{mq['val_acc'].iloc[-1]:.4f}（差 {d:.4f}）")
info = json.load(open(os.path.join(R_, "run_info.json"), encoding="utf-8"))
ok([s["resumed"] for s in info["sessions"]] == [False, True], "run_info.json には、再開のたびにセッションが追記される")

## 3. 比較用の実行（手順 4・9）

教科書では、Mac で `--epochs 4 --n-train 8000 --ckpt-dir runs/mac_base` を回し、Colab のセル5で同じ条件（`--ckpt-dir "$RUN_DIR" --ckpt-every-min 5 --max-hours 2`）を回して比べます。ここでは小さくして `--epochs 3 --n-train 4000` にします（`compare_runs.py` は3エポック以上あると最初のエポックをウォームアップとして除くので、3エポックにしてあります）。確かめること（手順 4）：「4エポック分の samples_per_s が記録される」（ここでは3エポック分）。

In [ ]:
# 教科書の設定：--epochs 4 --n-train 8000（Colab のセル5 では --ckpt-every-min 5 --max-hours 2 も付ける）
BASE = fresh("nb_base")                       # 教科書では runs/mac_base（Colab では Drive の runs/colab_toy01）
t0 = time.time()
run(f"train_toy.py --epochs 3 --n-train 4000 --ckpt-dir {BASE} --workers {WORKERS} --ckpt-every-min 5 --max-hours 2")
print(f"（{time.time() - t0:.0f} 秒）")
mb = pd.read_csv(os.path.join(BASE, "metrics.csv"))
print(mb[["epoch", "train_loss", "val_acc", "samples_per_s", "resumed"]])   # 教科書のセル6 と同じ列
ok(len(mb) == 3 and mb["samples_per_s"].notna().all(), "3エポック分の samples_per_s が記録された")

## 4. 実行どうしを比べる（手順 12、`compare_runs.py`）

教科書では `compare_runs.py runs/mac_base runs/colab_toy01` で、Mac と Colab の GPU を比べます。このノートブックでは同じ機械の2つの実行（3 節の `nb_base` と 1 節の `nb_quick`）を比べるので、比はほぼ 1 倍になるはずです。Colab の GPU で動かしているなら、Mac で回した `runs/mac_base` を左のファイルの欄から `runs/` に上げて、`run(f"compare_runs.py runs/mac_base {BASE}")` とすれば教科書と同じ比べになります（執筆時には確かめていません）。

注意（手順 12）：「この区間は、1回の実行の中のエポックを反復とみなした『この2回の実行』についての目安です。エポックどうしは同じマシン・同じ実行の中の値なので、独立な反復ではありません（疑似反復）」。

In [ ]:
out = run(f"compare_runs.py {BASE} {Q}")
ok(re.search(r"は nb_base の [\d.]+ 倍速い（95%区間 [\d.]+〜[\d.]+ 倍、Welch 自由度", out) is not None,
   "処理速度の比と 95% 区間（対数の差の Welch の t 区間）が出た")
ok(os.path.exists("compare_runs.png"), "学習曲線と処理速度の図 compare_runs.png ができた")
display(Image(filename="compare_runs.png", width=760))

## 5. チェックポイントの間隔：Young の近似式（III-2-3-2、発展課題）

保存に $\delta$ 秒かかり、切断が平均 $M$ 秒に1回（指数分布）起こるとき、むだになる時間の割合はおよそ $\delta/\tau + \tau/(2M)$ で、最小になる間隔は $\tau^\ast \approx \sqrt{2\delta M}$ です。本文：「$\delta = 10$ 秒、$M = 2$ 時間（7200秒）なら、$\tau^\ast \approx 380$ 秒」、図III-2-1：「τ* ≈ 380 秒で最小（約 5.3%）。スクリプトの初期値 5 分では 5.4%」。

発展課題の1つ目のとおり、切断を指数分布の乱数で起こすシミュレーションでも確かめます（1区間 = τ 秒の計算 ＋ δ 秒の保存。区間の途中で切れたら、その区間をやり直す）。

In [ ]:
import numpy as np
delta, M = 10.0, 7200.0
tau_star = (2 * delta * M) ** 0.5
f = lambda tau: delta / tau + tau / (2 * M)
ok(abs(tau_star - 380) < 1, f"τ* = √(2 × {delta:g} × {M:g}) = {tau_star:.0f} 秒（約 6 分）")
ok(abs(f(tau_star) * 100 - 5.3) < 0.05 and abs(f(300) * 100 - 5.4) < 0.05,
   f"むだの割合：τ* で {f(tau_star) * 100:.2f} %、5 分（300 秒）で {f(300) * 100:.2f} %（ほとんど差がない）")

rng = np.random.default_rng(0)
taus = np.array([60, 120, 200, 300, 380, 500, 700, 1000, 1500])
waste = []
for tau in taus:
    L = tau + delta                                  # 1区間の長さ（計算＋保存）
    q = np.exp(-L / M)                               # 1回の試みで区間を終えられる確率
    n_seg = 20000
    k = rng.geometric(q, n_seg) - 1                  # 区間ごとの、切れてやり直した回数
    # 切れるまでの時間は「L より短い指数分布」（逆関数法）
    u = rng.random(k.sum())
    lost = -M * np.log(1 - u * (1 - q))
    total = n_seg * L + lost.sum()
    waste.append(1 - n_seg * tau / total)
waste = np.array(waste)
for t, w in zip(taus, waste):
    print(f"  τ = {t:5.0f} 秒：むだ {w * 100:5.2f} %（近似式 {f(t) * 100:5.2f} %）")
best = taus[waste.argmin()]
ok(200 <= best <= 700 and abs(waste.min() * 100 - 5.3) < 0.8,
   f"シミュレーションでも、むだがいちばん小さいのは τ = {best} 秒付近（谷は平らで、√(2δM) ≈ 380 秒に近い）")

## 6. 結果の確認とランタイムの解放（手順 11、セル6・7）

教科書のセル6で metrics.csv を表示し、結果が Drive にあることを確かめてから、セル7の `runtime.unassign()` でランタイムを解放します（GPU の割り当てを返す。有料プランならユニットの消費も止まる）。このノートブックでは、うっかり解放しないように `RELEASE = False` にしてあります。Colab で使い終わったら、**結果を持ち帰ってから** `True` にして実行してください（Drive に保存していないなら、`runs/` は左のファイルの欄からダウンロードします）。

最後に、ノートブックを保存・共有する前に、出力をすべて消してください（III-2-6 の ⚠）。

In [ ]:
for name in ("nb_quick", "nb_resume", "nb_base"):
    m = pd.read_csv(os.path.join(RUNS, name, "metrics.csv"))
    print(f"{name:10s}", m[["epoch", "val_acc", "samples_per_s", "resumed"]].to_dict("list"))
RELEASE = False                                   # Colab で、結果を持ち帰ってから True にする
if IN_COLAB and RELEASE:
    from google.colab import runtime
    runtime.unassign()
else:
    print("ランタイムは解放していません（Colab では、結果を持ち帰ってから RELEASE = True で解放する）")

## まとめ

- `--quick` の2エポック目の val_acc が 0.9 以上（本文の目安 0.98 前後）になること、途中で SIGINT を送っても「チェックポイントから再開」で最後まで進み、止めなかった実行とほぼ同じ結果になること、Young の近似式の値（約 380 秒、5.3 %・5.4 %）を確かめました。
- 自分の private リポジトリで回すときは、教科書のセル1〜7（トークンはシークレットから読み、表示しない。clone のあとリモートの URL からトークンを消す）を使ってください。
- このノートブックに入れていないもの：GitHub のトークンとシークレット、private リポジトリの clone、本物の切断、Drive for desktop での持ち帰り。